# Trade Route Cost Classification System

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

# Load the dataset
print("Loading the dataset...")
df = pd.read_csv("../../../data/imputed_full_matrix_at_centroid.csv")

# Display basic information
print(f"Dataset shape: {df.shape}")
print(df.head())

# Check for missing values in the cost column
print(f"Missing values in 'Unit logistics costs ($/ton)': {df['Unit logistics costs ($/ton)'].isna().sum()}")

# Basic statistics of the cost column
print(df['Unit logistics costs ($/ton)'].describe())



Loading the dataset...
Dataset shape: (9512578, 15)
   origin_city  destination_city origin_ISO destination_ISO  year  \
0            1                 3        JPN             MYS  2020   
1            1                 3        JPN             MYS  2020   
2            1                 3        JPN             MYS  2020   
3            1                 3        JPN             MYS  2020   
4            1                 3        JPN             MYS  2020   

   commodity_index  mode  container_type     ship_type  flow(tonne)  \
0                0     0               3  Bulk carrier  1180.704383   
1                1     0               2        Tanker     0.000000   
2                2     0               3  Bulk carrier    61.802172   
3                5     0               0     Container  1432.441522   
4                6     0               3     Container     3.056417   

   distance(km) Mode_name              IFM_HS  Unit logistics costs ($/ton)  \
0   7020.525339       Air  

In [7]:
# Plot histogram of logistics costs
plt.figure(figsize=(12, 6))
sns.histplot(df['Unit logistics costs ($/ton)'], bins=50, kde=True)
plt.title('Distribution of Unit Logistics Costs', fontsize=16)
plt.xlabel('Unit Logistics Costs ($/ton)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.axvline(df['Unit logistics costs ($/ton)'].median(), color='red', linestyle='--', label=f'Median: {df["Unit logistics costs ($/ton)"].median():.2f}')
plt.axvline(df['Unit logistics costs ($/ton)'].mean(), color='green', linestyle='--', label=f'Mean: {df["Unit logistics costs ($/ton)"].mean():.2f}')
plt.legend()
plt.savefig('cost_distribution.png')
plt.close()

# Check for outliers with a boxplot
plt.figure(figsize=(12, 4))
sns.boxplot(x=df['Unit logistics costs ($/ton)'])
plt.title('Boxplot of Unit Logistics Costs', fontsize=16)
plt.xlabel('Unit Logistics Costs ($/ton)', fontsize=14)
plt.tight_layout()
plt.savefig('cost_boxplot.png')
plt.close()

print("\nApproach 1: Classification by Percentiles")
# Define thresholds using percentiles
low_threshold = df['Unit logistics costs ($/ton)'].quantile(0.333)
high_threshold = df['Unit logistics costs ($/ton)'].quantile(0.667)

# Create a new column with cost categories
df['cost_category_percentile'] = pd.cut(
    df['Unit logistics costs ($/ton)'],
    bins=[float('-inf'), low_threshold, high_threshold, float('inf')],
    labels=['Low Cost', 'Moderate Cost', 'High Cost']
)

# Display category distribution
print("Cost Category Distribution (Percentile-based):")
category_counts = df['cost_category_percentile'].value_counts()
print(category_counts)
print(f"\nCategory Percentages:\n{(category_counts / len(df) * 100).round(2)}%")

# Display threshold values
print(f"\nLow Cost: Less than ${low_threshold:.2f} per ton")
print(f"Moderate Cost: ${low_threshold:.2f} to ${high_threshold:.2f} per ton")
print(f"High Cost: Greater than ${high_threshold:.2f} per ton")

# Visualize the categories
plt.figure(figsize=(12, 6))
sns.histplot(
    data=df, 
    x='Unit logistics costs ($/ton)', 
    hue='cost_category_percentile',
    bins=50, 
    multiple='stack',
    palette='viridis'
)
plt.axvline(low_threshold, color='red', linestyle='--', label=f'Low Threshold: ${low_threshold:.2f}')
plt.axvline(high_threshold, color='green', linestyle='--', label=f'High Threshold: ${high_threshold:.2f}')
plt.title('Distribution of Logistics Costs by Category (Percentile-based)', fontsize=16)
plt.xlabel('Unit Logistics Costs ($/ton)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.legend()
plt.savefig('percentile_distribution.png')
plt.close()

# Bar chart of category counts
plt.figure(figsize=(10, 6))
sns.countplot(x='cost_category_percentile', data=df, palette='viridis')
plt.title('Number of Routes by Cost Category (Percentile-based)', fontsize=16)
plt.xlabel('Cost Category', fontsize=14)
plt.ylabel('Number of Routes', fontsize=14)
plt.savefig('percentile_counts.png')
plt.close()


Approach 1: Classification by Percentiles
Cost Category Distribution (Percentile-based):
cost_category_percentile
Moderate Cost    3177165
Low Cost         3167728
High Cost        3167685
Name: count, dtype: int64

Category Percentages:
cost_category_percentile
Moderate Cost    33.4
Low Cost         33.3
High Cost        33.3
Name: count, dtype: float64%

Low Cost: Less than $201.68 per ton
Moderate Cost: $201.68 to $2693.84 per ton
High Cost: Greater than $2693.84 per ton


In [8]:
print("\nApproach 2: Classification using K-means Clustering")
# Prepare data for clustering
# Get only the cost column and handle missing values if any
X = df[['Unit logistics costs ($/ton)']].copy()
X = X.fillna(X.mean())

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply K-means clustering with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cost_category_kmeans'] = kmeans.fit_predict(X_scaled)

# Get cluster centers and transform back to original scale
cluster_centers = scaler.inverse_transform(kmeans.cluster_centers_)

# Sort clusters by their centers to maintain consistent labels (Low, Moderate, High)
sorted_indices = np.argsort(cluster_centers.flatten())
mapping = {sorted_indices[0]: 'Low Cost', sorted_indices[1]: 'Moderate Cost', sorted_indices[2]: 'High Cost'}
df['cost_category_kmeans'] = df['cost_category_kmeans'].map(mapping)

# Display cluster centers
print("K-means Cluster Centers (in original scale):")
for i, center in enumerate(cluster_centers.flatten()[sorted_indices]):
    category = ['Low Cost', 'Moderate Cost', 'High Cost'][i]
    print(f"{category}: ${center:.2f} per ton")

# Display category distribution
print("\nCost Category Distribution (K-means):")
category_counts_kmeans = df['cost_category_kmeans'].value_counts()
print(category_counts_kmeans)
print(f"\nCategory Percentages:\n{(category_counts_kmeans / len(df) * 100).round(2)}%")

# Visualize the K-means categories
plt.figure(figsize=(12, 6))
sns.histplot(
    data=df, 
    x='Unit logistics costs ($/ton)', 
    hue='cost_category_kmeans',
    bins=50, 
    multiple='stack',
    palette='viridis'
)

# Add vertical lines for cluster centers
for center in cluster_centers.flatten()[sorted_indices]:
    plt.axvline(center, color='red', linestyle='--')
    
plt.title('Distribution of Logistics Costs by Category (K-means)', fontsize=16)
plt.xlabel('Unit Logistics Costs ($/ton)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.legend()
plt.savefig('kmeans_distribution.png')
plt.close()

# Bar chart of K-means category counts
plt.figure(figsize=(10, 6))
category_order = ['Low Cost', 'Moderate Cost', 'High Cost']
sns.countplot(x='cost_category_kmeans', data=df, order=category_order, palette='viridis')
plt.title('Number of Routes by Cost Category (K-means)', fontsize=16)
plt.xlabel('Cost Category', fontsize=14)
plt.ylabel('Number of Routes', fontsize=14)
plt.savefig('kmeans_counts.png')
plt.close()


Approach 2: Classification using K-means Clustering
K-means Cluster Centers (in original scale):
Low Cost: $2045.70 per ton
Moderate Cost: $27680.04 per ton
High Cost: $80490.21 per ton

Cost Category Distribution (K-means):
cost_category_kmeans
Low Cost         8402145
Moderate Cost     880470
High Cost         229963
Name: count, dtype: int64

Category Percentages:
cost_category_kmeans
Low Cost         88.33
Moderate Cost     9.26
High Cost         2.42
Name: count, dtype: float64%


No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


In [ ]:
# Add after K-means clustering section
from sklearn.metrics import silhouette_score

# Calculate silhouette score for K-means clustering
silhouette_avg = silhouette_score(X_scaled, kmeans.labels_)
print(f"Silhouette Score for K-means clustering: {silhouette_avg:.4f}")
# Values close to 1 indicate well-separated clusters